<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/08_score_storm_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Storm-identification probe: which storms can each model name?

Reads every identification results file in `results/storm_identification/dev/` and `results/storm_identification/test/` (subfolders `claude/` and `openai/`) (any model, with or without location), checks each answer against the true storm, and writes two tables per set to `results/analysis/dev/` and `results/analysis/test/`:

- `storm_identification_<set>_summary.csv`: per model and probe version, how many cases were identified, how many named storms, how often the model said "unknown", and its confidence when right and when wrong;
- `storm_identification_<set>_identified_storms.csv`: every storm that at least one model identified, with the first observation at which each model got it right and on how many of the storm's cases.

**Rules.**
- A case is **identified** when the model gives the storm's name **and** season. Names are reused across years and basins, so the name alone is not enough.
- The "name only" column is shown for reference.
- An answer of "UNNAMED" for a storm whose name is UNNAMED never counts as identified. It names nothing.
- A **storm is identified** by a model when at least one of its cases is.
- Names are compared after `parse_identify_response`, which:
  - upper-cases the name;
  - removes words such as "TROPICAL STORM" before it;
  - reads the basin "CP" as EP.
- Names are compared without hyphens or spaces, so "KONG REY" matches "KONG-REY".
- An answer that can't be parsed counts as not identified and is listed in the "unparseable answers" column.

**Results files** must be named `<model>_detailed_results_<case file name>.jsonl.gz`, as notebooks 07a and 07b write them. New models are picked up automatically.

## 1. Setup

Works out the repo folder and makes `src/` importable, as in the other notebooks. In Colab it first clones this repo.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

## 2. Read every results file and score it

The truth (name, season, basin) comes from the matching case file in `model_inputs/storm_identification/`. Every results file must answer every case in its case file, exactly once.

In [ ]:
import pandas as pd
from prompts import name_key, parse_identify_response, read_answers, read_records

RESULTS_DIR = ROOT / "results" / "storm_identification"
ANALYSIS_DIR = ROOT / "results" / "analysis"

scored = []
for path in sorted(RESULTS_DIR.glob("*/*/*_detailed_results_*.jsonl.gz")):    # dev/claude/, test/openai/, ...
    model, case_name = path.name.removesuffix(".jsonl.gz").split("_detailed_results_")
    case_file = ROOT / "model_inputs" / "storm_identification" / path.parent.parent.name / f"{case_name}.jsonl.gz"
    truth = pd.DataFrame(read_records(case_file)).drop(columns="prompt")
    answers = read_answers(path, parse_identify_response).drop(columns="model")
    assert set(answers["case_id"]) == set(truth["case_id"]), f"{path.name} does not answer exactly the cases in {case_file.name}"

    d = truth.merge(answers, on="case_id", validate="one_to_one", suffixes=("", "_pred"))
    d["model"], d["set"] = model, truth["split"].iloc[0]
    named = d["name"] != "UNNAMED"
    d["name_ok"] = d["parse_ok"] & named & (d["storm_name"].map(name_key) == d["name"].map(name_key))
    d["name_season_ok"] = d["name_ok"] & (d["season_pred"] == d["season"])
    d["unknown"] = d["parse_ok"] & (d["storm_name"] == "unknown")
    scored.append(d)
    print(f"{path.name}: {len(d)} cases, {int((~d['parse_ok']).sum())} unparseable")

scored = pd.concat(scored, ignore_index=True)

## 3. Summary per model and probe version

`named storms identified` counts storms with a real name (not UNNAMED) that the model identified on at least one case. Confidence is averaged over the answers that identified the storm, and over the wrong answers that were not "unknown".

In [ ]:
def summary_row(d):
    n_named = d.loc[d["name"] != "UNNAMED", "sid"].nunique()
    storms_identified = d[d["name"] != "UNNAMED"].groupby("sid")["name_season_ok"].any().sum()
    right = d.loc[d["name_season_ok"], "confidence"].mean()
    wrong = d.loc[d["parse_ok"] & ~d["name_season_ok"] & ~d["unknown"], "confidence"].mean()
    return {
        "cases correct, name only": f"{int(d['name_ok'].sum())} ({d['name_ok'].mean():.1%})",
        "cases correct, name + season": f"{int(d['name_season_ok'].sum())} ({d['name_season_ok'].mean():.1%})",
        "named storms identified": f"{int(storms_identified)}/{n_named}",
        "answered unknown": f"{d['unknown'].mean():.1%}",
        "unparseable answers": int((~d["parse_ok"]).sum()),
        "mean confidence correct / wrong": f"{'-' if pd.isna(right) else f'{right:.2f}'} / {wrong:.2f}",
    }

for set_name, s in scored.groupby("set"):
    table = pd.DataFrame([{"model": m, "probe": p.removeprefix("identify_"), **summary_row(d)}
                          for (m, p), d in s.groupby(["model", "probe"])])
    (ANALYSIS_DIR / set_name).mkdir(parents=True, exist_ok=True)
    table.to_csv(ANALYSIS_DIR / set_name / f"storm_identification_{set_name}_summary.csv", index=False)
    print(f"== {set_name} ==")
    display(table)

## 4. Which storms were identified, and when

One row per storm identified by at least one model and probe version. Each cell gives the first observation at which that model named the storm correctly, and on how many of the storm's cases it did (`obs 7 (3/41)` = first at observation 7, right on 3 of its 41 cases). The probe versions are `noloc` (no location) and `loc` (with location).

In [ ]:
for set_name, s in scored.groupby("set"):
    hits = s[s["name_season_ok"]]
    storms = (s.drop_duplicates("sid").set_index("sid")[["name", "basin", "storm_group"]]
              .rename(columns={"storm_group": "group"}))
    storms["n_cases"] = s.groupby("sid")["case_id"].nunique()
    table = storms.loc[sorted(hits["sid"].unique(), key=lambda sid: storms.at[sid, "name"])].copy()
    for (m, p), d in s.groupby(["model", "probe"]):
        h = d[d["name_season_ok"]].groupby("sid")["obs_index"].agg(["min", "size"])
        col = f"{m} {'noloc' if p == 'identify_no_location' else 'loc'}"
        table[col] = [f"obs {h.at[sid, 'min']} ({h.at[sid, 'size']}/{storms.at[sid, 'n_cases']})" if sid in h.index else ""
                      for sid in table.index]
    table = table.reset_index()
    table.to_csv(ANALYSIS_DIR / set_name / f"storm_identification_{set_name}_identified_storms.csv", index=False)
    print(f"== {set_name}: {len(table)} storms identified by at least one model ==")
    display(table)

## Notes

- **Cases that can't be parsed** count as not identified. Their number is in the summary table, so it can be reported.